# Tracing Basics

### Setup

Make sure you set your environment variables, including your OpenAI API key.

In [1]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv()

True

### Tracing with @traceable

The @traceable decorator is a simple way to log traces from the LangSmith Python SDK. Simply decorate any function with @traceable.

The decorator works by creating a run tree for you each time the function is called and inserting it within the current trace. The function inputs, name, and other information is then streamed to LangSmith. If the function raises an error or if it returns a response, that information is also added to the tree, and updates are patched to LangSmith so you can detect and diagnose sources of errors. This is all done on a background thread to avoid blocking your app's execution.

In [ ]:
import os
import nest_asyncio
from typing import List
from openai import OpenAI
from langsmith import traceable
from utils import get_vector_db_retriever


# --- Stack Configuration (Crusty Nails Zero-Dollar Setup) ---
MODEL_PROVIDER = "groq"
MODEL_NAME = "openai/gpt-oss-120b"  # Quota fallback: "qwen/qwen3.8-27b"
APP_VERSION = 1.0
RAG_SYSTEM_PROMPT = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the latest question in the conversation. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.
"""

# Route standard OpenAI client to Groq's endpoint
openai_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

nest_asyncio.apply()
retriever = get_vector_db_retriever()

# --- Tracing Hierarchy ---

@traceable(name="retrieve_documents", run_type="retriever")
def retrieve_documents(question: str):
    # LangChain retriever automatically traces internally, 
    # but wrapping gives us a clean span boundary in the tree.
    return retriever.invoke(question)


@traceable(name="generate_response")
def generate_response(question: str, documents):
    # Rule 5: Token Diet - slice to [:600] to prevent payload ballooning on free tiers
    formatted_docs = "\n\n".join(doc.page_content[:600] for doc in documents)
    messages = [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"Context: {formatted_docs} \n\n Question: {question}"
        }
    ]
    return call_openai(messages)


@traceable(name="call_groq_llm", run_type="llm")
def call_openai(
    messages: List[dict], model: str = MODEL_NAME, temperature: float = 0.0
):
    return openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )


@traceable(name="langsmith_rag", run_type="chain")
def langsmith_rag(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content

[1/3] 🔍 Locating union.parquet...
[2/3] 📖 Found 'resources\union.parquet'. Reading parquet into DataFrame...
      Loaded 2187 document chunks from parquet.
      Using text column: 'texts', metadata column: 'metadatas'
[3/3] ⚡ Building local TF-IDF index across 2187 documents...
✅ Retriever ready in 1.65 seconds! Zero API calls, zero web scraping.


@traceable handles the RunTree lifecycle for you!

In [3]:
question = "How can I trace with the @traceable decorator?"
ai_answer = langsmith_rag(question)
print(ai_answer)

Apply the @traceable decorator to any Python function you want to monitor (e.g., @traceable def my_func(...): …). When you call that function—using await if it’s async—LangSmith automatically creates a run and propagates the trace through any nested calls. The traces are then logged to your LangSmith project without additional instrumentation.


##### Let's take a look in LangSmith!

### Adding Metadata

LangSmith supports sending arbitrary metadata along with traces.

Metadata is a collection of key-value pairs that can be attached to runs. Metadata can be used to store additional information about a run, such as the version of the application that generated the run, the environment in which the run was generated, or any other information that you want to associate with a run. Similar to tags, you can use metadata to filter runs in the LangSmith UI, and can be used to group runs together for analysis.

In [ ]:
from langsmith import traceable

@traceable(
    metadata={"vectordb": "sklearn"}
)

def retrieve_documents(question: str):
    return retriever.invoke(question)

@traceable
def generate_response(question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    messages = [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"Context: {formatted_docs} \n\n Question: {question}"
        }
    ]
    return call_openai(messages)

@traceable(
    metadata={"model_name": MODEL_NAME, "model_provider": MODEL_PROVIDER}
)

def call_openai(
    messages: List[dict], model: str = MODEL_NAME, temperature: float = 0.0
) -> str:
    return openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )

@traceable
def langsmith_rag(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content


In [8]:
question = "How do I add Metadata to a Run with @traceable?"
ai_answer = langsmith_rag(question)
print(ai_answer)

Add static metadata when you decorate the function, e.g. `@traceable(metadata={"llm": "gpt‑4.1‑mini"})` or include other keys in the decorator’s arguments. You can also set metadata dynamically inside the function by getting the current run tree (`rt = ls.get_current_run_tree()`) and updating `rt.metadata["key"] = "value"`. Finally, you can supply metadata at call time via the `langsmith_extra` argument (`langsmith_extra={"metadata": {"key": "value"}}`).


You can also add metadata at runtime!

In [9]:
question = "How do I add metadata at runtime?"
ai_answer = langsmith_rag(question, langsmith_extra={"metadata": {"runtime_metadata": "foo"}})
print(ai_answer)

You can attach metadata at runtime by supplying a `RunnableConfig` when you invoke a runnable, e.g.:

```python
chain.invoke(
    {"input": "What is the meaning of life?"},
    {"metadata": {"invoke-key": "invoke‑value"}, "tags": ["invoke-tag"]},
)
```

If you’re using OpenTelemetry spans directly, start a span and call `span.set_attribute("your.key", "value")` inside the span’s context. Both methods add the metadata to the trace for that execution.


##### Let's take a look in LangSmith!